In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV

# 1. Mock Data Generation reflecting Media User Behaviors
np.random.seed(42)
n_samples = 15000
data = pd.DataFrame({
    'days_since_last_login': np.random.exponential(scale=7, size=n_samples),
    'video_completion_rate': np.random.uniform(0.1, 1.0, size=n_samples),
    'monthly_active_hours': np.random.gamma(shape=3, scale=4, size=n_samples),
    'cross_genre_views': np.random.randint(1, 8, size=n_samples),
    'converted': np.random.choice([0, 1], size=n_samples, p=[0.88, 0.12])
})

# Inject synthetic relationships to ensure the model can learn patterns
data.loc[data['days_since_last_login'] < 2, 'converted'] = \
    np.random.choice([0, 1], size=len(data[data['days_since_last_login'] < 2]), p=[0.6, 0.4])

X = data.drop(columns=['converted'])
y = data['converted']

# 2. Stratified Data Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. Model Initialization and Training
xgb_base = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    eval_metric='logloss',
    random_state=42
)

# Apply calibration to ensure predicted probabilities match real world outcomes
calibrated_model = CalibratedClassifierCV(estimator=xgb_base, method='sigmoid', cv=3)
calibrated_model.fit(X_train, y_train)

# 4. Predictions and Performance Evaluation
propensity_scores = calibrated_model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, propensity_scores)
brier_score = brier_score_loss(y_test, propensity_scores)

print(f"Model Performance Evaluation Metrics:")
print(f"--------------------------------------")
print(f"Area Under the ROC Curve (AUC): {auc_score:.4f}")
print(f"Brier Score (Calibration Loss): {brier_score:.4f}")


Model Performance Evaluation Metrics:
--------------------------------------
Area Under the ROC Curve (AUC): 0.6693
Brier Score (Calibration Loss): 0.1425
